# DAY 3 — 멀티모달 기반 비정형 데이터 정보화

**AI 적용을 위한 현장 데이터 수집 및 디지털화** · DAY 3 실습

교재 PART 03 (pp.128–178) 의 실습 코드를 코랩에서 바로 돌릴 수 있게 옮긴 것입니다.
설명과 그림은 교재와 학습사이트에 있습니다 — https://build-data.jobability.co.kr

---
### 코랩 쓰는 법 세 가지만

1. **셀 실행** — 코드 칸 왼쪽 ▶ 를 누르거나 `Shift + Enter`.
2. **위에서부터 차례로** — 앞 칸에서 만든 것을 뒤 칸이 씁니다. 건너뛰면 오류가 납니다.
3. **내 사본으로** — 「파일 → 드라이브에 사본 저장」을 먼저 해야 고친 내용이 남습니다.

## 0. 실습 준비

이 아래 두 칸은 세션을 새로 열 때마다 한 번씩 실행합니다.

In [ ]:
# 그래프에 한글이 네모로 나오지 않게 폰트를 깝니다. 세션마다 한 번만 하면 됩니다.
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
import glob, os
cand = glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf") + glob.glob("fonts/NanumGothic*.ttf")
if cand:
    font_manager.fontManager.addfont(cand[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=cand[0]).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", plt.rcParams["font.family"])

In [ ]:
# 실습 데이터 준비
# 1) 왼쪽 폴더 아이콘에 data_day3_docs.zip 을 올려 두었으면 그것을 풉니다.
# 2) 없으면 파일 선택 창이 열립니다 — 받은 data_day3_docs.zip 을 고르세요.
import os, zipfile
if not os.path.exists("checklists"):
    if os.path.exists("data_day3_docs.zip"):
        zipfile.ZipFile("data_day3_docs.zip").extractall(".")
    else:
        try:
            from google.colab import files
            print("data_day3_docs.zip 이 없습니다. 파일 선택 창에서 올려 주세요.")
            files.upload()
            zipfile.ZipFile("data_day3_docs.zip").extractall(".")
        except ImportError:
            raise SystemExit("data_day3_docs.zip 을 이 폴더에 두고 다시 실행하세요.")
print("준비된 폴더:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

## 2. 음향 AI와 멀티모달 LLM의 원리

### 음향 신호의 디지털 변환 원리

## 4. 실습 A(비전): 결함 사진 자동 라벨링

### 결함 사진 라벨링 부재 문제

**실습 개요**

### 단계 1~2: 환경과 데이터 준비

**단계 1. Colab과 API 키 준비**

**[3-2] API 키 입력받기**

In [ ]:
from getpass import getpass
API_KEY = (os.environ.get("GEMINI_API_KEY") or getpass("Gemini API Key 입력: ")).strip()

**[3-3] Gemini 이미지 호출 함수**

In [ ]:
import base64, json, urllib.request
MODEL = "gemini-3.5-flash"
def gemini_image(prompt, file_path):
    data = base64.b64encode(open(file_path, "rb").read()).decode()
    body = {"contents": [{"parts": [
        {"inline_data": {"mime_type": "image/jpeg", "data": data}},
        {"text": prompt}]}],
        "generationConfig": {"temperature": 0}}
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json",
                                          "x-goog-api-key": API_KEY,        # 키는 URL이 아니라 헤더로(노출방지)
                                          "User-Agent": "Mozilla/5.0 (data-pipeline-practice)"})
    resp = json.load(urllib.request.urlopen(req, timeout=180))
    text = resp["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(text.strip().removeprefix("```json").removesuffix("```").strip("` \n"))
print("호출 준비 완료 — 모델:", MODEL)

**단계 2. casting 데이터 내려받기**

**[3-4] casting 데이터 내려받기**

In [ ]:
import kagglehub, os, glob
path = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
base = os.path.join(path, "casting_512x512", "casting_512x512")
def_files = sorted(glob.glob(os.path.join(base, "def_front", "*.jpeg")))
ok_files  = sorted(glob.glob(os.path.join(base, "ok_front", "*.jpeg")))
print("결함(def)", len(def_files), "장 / 정상(ok)", len(ok_files), "장")

### 단계 3~4: 육안 확인과 단건 라벨링

**단계 3. 사진 육안으로 먼저 확인하기**

**[3-6] 정상 ·결함 사진 나란히 보기**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, f, tag in [(axes[0], ok_files[0], "ok (정상)"), (axes[1], def_files[0], "def (결함)")]:
    ax.imshow(Image.open(f)); ax.set_title(tag); ax.axis("off")
plt.tight_layout(); plt.show()

**단계 4. 단건 라벨링 요청하기**

**[3-7] 자연어 지시: 결함 판정 라벨링**

```
이 주조품 사진에 표면 결함이 있으면 def, 없으면 ok로 판정하고
근거를 한 문장으로 JSON에 담아줘.
이 지시를 모델에 보낼 프롬프트로 다듬으면 다음과 같습니다.
```

**[3-8] 프롬프트: 주조품 표면 결함 판정**

```
이 사진은 엔진 냉각수 펌프 임펠러 주조품의 입고 검사 사진입니다.
표면 결함(기공, 균열, 미성형, 버, 흠집 등)이 있는지 판정하세요.
{"라벨": "ok|def", "근거": "한 문장"} JSON만 출력하세요. 결함이 있으면 def, 없으면 ok.
프롬프트를 뜯어보면 네 가지가 들어 있습니다.
```

**[3-9] 사진 한 장 라벨링**

In [ ]:
PROMPT = '''이 사진은 엔진 냉각수 펌프 임펠러 주조품의 입고 검사 사진입니다.
표면 결함(기공, 균열, 미성형, 버, 흠집 등)이 있는지 판정하세요.
{"라벨": "ok|def", "근거": "한 문장"} JSON만 출력하세요. 결함이 있으면 def, 없으면 ok.'''
r = gemini_image(PROMPT, def_files[0])
print(r)

### 단계 5~6: 60장 일괄 라벨링과 혼동 행렬

**단계 5. 결함 30장·정상 30장 일괄 라벨링**

**[3-10] 결함 30장 ·정상 30장 일괄 라벨링**

In [ ]:
import time
results = []
for true, files in [("def", def_files[:30]), ("ok", ok_files[:30])]:
    for f in files:
        try:
            r = gemini_image(PROMPT, f)
            pred = r.get("라벨")
        except Exception:
            r, pred = {}, None
        results.append({"file": os.path.basename(f), "true": true,
                        "pred": pred, "근거": r.get("근거", "")})
        time.sleep(0.4)
print("라벨링 완료:", len(results), "장")

**단계 6. 혼동 행렬로 채점하기**

**[3-11] 혼동 행렬로 채점하기**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_true = [r["true"] for r in results]
y_pred = [r["pred"] or "?" for r in results]

### 단계 7: 오분류 재검과 검토 규칙 수립

**단계 7. 오분류 재검**

**[3-13] 오분류 사진 재검**

In [ ]:
miss = [r for r in results if r["true"] != r["pred"]]
print(f"오분류 {len(miss)}건")
if miss:
    fig, axes = plt.subplots(1, min(3, len(miss)), figsize=(11, 4))
    axes = [axes] if len(miss) == 1 else list(axes)
    lookup = {os.path.basename(f): f for f in def_files[:30] + ok_files[:30]}
    for ax, m in zip(axes, miss[:3]):
        ax.imshow(Image.open(lookup[m["file"]]))
        ax.set_title(f"{m['true']} → {m['pred']}", fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()
    for m in miss[:3]:
        print(f"- {m['file']}: {m['근거'][:60]}")

## 5. 실습 B(음향): 가동음 정상·이상 분류

### 설비 음향 데이터 분석의 필요성

**실습 개요**

### 단계 1~2: 데이터 준비와 파형 확인

**단계 1. 음향 데이터 준비**

**[3-15] 음향 데이터 압축 해제**

In [ ]:
import os, zipfile
if not os.path.exists("audio"):
    zipfile.ZipFile("data_day3_audio.zip").extractall(".")
import glob
n_files = sorted(glob.glob("audio/normal/*.wav"))
a_files = sorted(glob.glob("audio/abnormal/*.wav"))
print("정상", len(n_files), "개 / 이상", len(a_files), "개")

**[3-17] librosa와 한글 폰트 준비**

In [ ]:
import librosa, librosa.display
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import glob
_font = (glob.glob("fonts/NanumGothic*.ttf")
         + glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"))
if _font:
    font_manager.fontManager.addfont(_font[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=_font[0]).get_name()
else:
    print("한글 폰트를 찾지 못했습니다 — 맨 위 「한글 폰트」 칸을 먼저 실행하세요.")
plt.rcParams["axes.unicode_minus"] = False
print("librosa", librosa.__version__)

**[3-19] 자연어 지시: 파형 나란히 그리기**

```
정상과 이상 펌프 wav를 하나씩 읽어 파형을 나란히 그려줘.
```

**[3-20] 정상 ·이상 파형 그리기**

In [ ]:
yn, sr = librosa.load(n_files[0], sr=None)
ya, _  = librosa.load(a_files[0], sr=None)
print(f"샘플레이트 {sr} Hz, 길이 {len(yn)/sr:.0f}초")
fig, axes = plt.subplots(2, 1, figsize=(10, 3.6), sharex=True)
axes[0].plot(np.arange(len(yn))/sr, yn, lw=0.3); axes[0].set_title("정상 펌프 — 파형")
axes[1].plot(np.arange(len(ya))/sr, ya, lw=0.3, color="tab:orange"); axes[1].set_title("이상 펌프 — 파형")
axes[1].set_xlabel("시간(초)")
plt.tight_layout(); plt.show()

### 단계 3: 스펙트로그램 시각 분석

**단계 3. 스펙트로그램과 평균 스펙트럼**

**[3-22] 자연어 지시: 스펙트로그램과 평균 스펙트럼**

```
두 소리의 멜 스펙트로그램을 나란히 그리고, 각 20개 파일의 평균 스펙트럼을
겹쳐 그려서 차이가 나는 주파수 대역을 찾아줘.
먼저 멜 스펙트로그램입니다.
```

**[3-23] 멜 스펙트로그램 그리기**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (y, tag) in zip(axes, [(yn, "정상"), (ya, "이상")]):
    M = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=96)
    img = librosa.display.specshow(librosa.power_to_db(M, ref=np.max), sr=sr,
                                   x_axis="time", y_axis="mel", ax=ax, cmap="magma", vmin=-70, vmax=0)
    ax.set_title(f"{tag} 펌프 — 멜 스펙트로그램"); ax.set_xlabel("시간(초)")
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.85)
plt.show()

**[3-24] 평균 스펙트럼 비교**

In [ ]:
def avg_spec(files):
    return np.mean([np.abs(librosa.stft(librosa.load(f, sr=None)[0], n_fft=2048)).mean(axis=1)
                    for f in files], axis=0)
n_avg, a_avg = avg_spec(n_files[:20]), avg_spec(a_files[:20])
freqs = np.linspace(0, sr/2, len(n_avg))
plt.figure(figsize=(10, 3.2))
plt.plot(freqs, 20*np.log10(n_avg), label="정상(20개 평균)")
plt.plot(freqs, 20*np.log10(a_avg), label="이상(20개 평균)")
plt.axvspan(2000, 4000, alpha=0.12, color="red")
plt.xlabel("주파수(Hz)"); plt.ylabel("크기(dB)"); plt.xlim(0, 8000)
plt.title("평균 스펙트럼 — 이상음은 2~4kHz 대역이 약 +6dB 높다")
plt.legend(); plt.tight_layout(); plt.show()

### 단계 4: MFCC 특징 추출

**단계 4. MFCC 특징 추출**

**[3-25] 자연어 지시: MFCC 특징 행렬 만들기**

```
모든 wav에서 MFCC 13개의 평균과 표준편차를 뽑아 특징 행렬 X와 라벨 y를 만들어줘.
```

**[3-26] MFCC 특징 추출**

In [ ]:
X, y = [], []
for label, files in [(0, n_files), (1, a_files)]:
    for f in files:
        sig, _ = librosa.load(f, sr=None)
        m = librosa.feature.mfcc(y=sig, sr=sr, n_mfcc=13)
        X.append(np.r_[m.mean(axis=1), m.std(axis=1)])
        y.append(label)
X, y = np.array(X), np.array(y)
print("특징 행렬:", X.shape, "(파일 200개 × 특징 26개)")

### 단계 5: 분류와 혼동 행렬

**단계 5. 정상·이상 자동 분류**

**[3-28] 자연어 지시: 분류와 혼동 행렬**

```
특징을 7:3으로 나눠 로지스틱 회귀로 학습하고, 테스트 정확도와 혼동 행렬을 보여줘.
```

**[3-29] 로지스틱 회귀 학습과 평가**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

## 6. 실습 C(문서·음성): 추출·검증·적재

### 문서·음성 데이터 3종 개요

**실습 개요**

### 점검표 사진의 JSON 구조화 추출

**단계 1. 번들과 API 키 준비**

**[3-33] 문서 번들 압축 해제**

In [ ]:
import os, zipfile
if not os.path.exists("checklists"):
    zipfile.ZipFile("data_day3_docs.zip").extractall(".")
print(sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

**[3-35] 문서 파이프라인 API 키 입력**

In [ ]:
from getpass import getpass
API_KEY = (os.environ.get("GEMINI_API_KEY") or getpass("Gemini API Key 입력: ")).strip()

**[3-36] Gemini 파일 ·프롬프트 호출 함수**

```
import base64, json, urllib.request
MODEL = "gemini-3.5-flash"
def gemini(prompt, file_path=None, mime=None):
"""파일(이미지·음성)과 프롬프트를 Gemini에 보내고 텍스트 응답을 받는다."""
parts = []
if file_path:
data = base64.b64encode(open(file_path, "rb").read()).decode()
parts.append({"inline_data": {"mime_type": mime, "data": data}})
parts.append({"text": prompt})
body = {"contents": [{"parts": parts}], "generationConfig": {"temperature": 0}}
url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
req = urllib.request.Request(url, data=json.dumps(body).encode(),
headers={"Content-Type": "application/json",
"x-goog-api-key": API_KEY,        # 키는 URL이 아니라 헤더로(노출
방지)
"User-Agent": "Mozilla/5.0 (data-pipeline-practice)"})
resp = json.load(urllib.request.urlopen(req, timeout=300))
return resp["candidates"][0]["content"]["parts"][0]["text"]
def as_json(text):
"""응답에서 JSON만 꺼낸다(코드 펜스 제거)."""
return json.loads(text.strip().removeprefix("```json").removesuffix("```").strip("` \n"))
```

**단계 2. 점검표 한 장 추출**

**[3-37] 점검표 사진 확인**

In [ ]:
from IPython.display import Image, display
display(Image("checklists/daily_001.png", width=520))

**[3-38] 자연어 지시: 점검표 항목 추출**

```
이 점검표 사진에서 장비ID·점검일자·점검자와 항목별 판정(양호/불량)·비고를
JSON으로 추출해줘.
```

**[3-39] 프롬프트: 일일점검표 JSON 추출**

```
이 사진은 굴착기 일일점검표입니다. 다음 JSON 형식으로 내용을 추출하세요.
{"장비ID": "", "점검일자": "YYYY-MM-DD", "점검자": "",
"항목": {"<항목명>": {"판정": "양호|불량", "비고": ""}}, "특이사항": ""}
항목명은 사진에 적힌 그대로, JSON만 출력하세요.
프롬프트에서 두 가지를 짚어 두겠습니다.
```

**[3-40] 점검표 한 장 추출**

In [ ]:
PROMPT_DAILY = '''이 사진은 굴착기 일일점검표입니다. 다음 JSON 형식으로 내용을 추출하세요.
{"장비ID": "", "점검일자": "YYYY-MM-DD", "점검자": "",
 "항목": {"<항목명>": {"판정": "양호|불량", "비고": ""}}, "특이사항": ""}
항목명은 사진에 적힌 그대로, JSON만 출력하세요.'''
ext = as_json(gemini(PROMPT_DAILY, "checklists/daily_001.png", "image/png"))
print(json.dumps(ext, ensure_ascii=False, indent=1)[:400], "'''")

**단계 3. 정답지로 채점하기**

**[3-41] 정답지와 대조해 채점하기**

In [ ]:
gt_all = {p["파일"]: p for p in json.load(open("checklists/checklists_ground_truth.json"))["사진"]}
def score_daily(file_name, ext):
    gt = gt_all[file_name]
    hit = sum(1 for k, v in gt["항목"].items()
              if (ext.get("항목") or {}).get(k, {}).get("판정") == v["판정"])
    head = [ext.get("장비ID") == gt["장비ID"], ext.get("점검일자") == gt["점검일자"],
            ext.get("점검자") == gt["점검자"]]
    return hit, len(gt["항목"]), sum(head)
hit, total, head = score_daily("daily_001.png", ext)
print(f"항목 판정 {hit}/{total} 일치, 헤더(장비·일자·점검자) {head}/3 일치")

**단계 4. 다섯 장 일괄 채점**

**[3-42] 점검표 5장 일괄 채점**

In [ ]:
import time
rows = []
for f in ["daily_001.png", "daily_002.png", "daily_003.png", "daily_004.png", "daily_005.png"]:
    e = as_json(gemini(PROMPT_DAILY, f"checklists/{f}", "image/png"))
    hit, total, head = score_daily(f, e)
    rows.append((f, hit, total, head))
    print(f"{f}: 항목 {hit}/{total}, 헤더 {head}/3")
    time.sleep(1)
acc = sum(r[1] for r in rows) / sum(r[2] for r in rows)
print(f"\n항목 판정 정확도: {acc:.1%}")

### 상담 녹음의 화자 구분 전사(STT)

**단계 5. 상담 녹음 전사**

**[3-43] 자연어 지시: 상담 녹음 화자 구분 전사**

```
이 상담 녹음을 한국어로 전사해줘. 발화자를 '상담원:'과 '고객:'으로 구분해서 줄 단위로.
```

**[3-44] 상담 녹음 전사**

In [ ]:
transcript = gemini('''이 음성은 건설장비 고객센터 상담 녹음입니다. 전체를 한국어로 전사하세요.
발화자를 '상담원:'과 '고객:'으로 구분해 줄 단위로 적으세요. 전사 텍스트만 출력하세요.''',
                    "audio/call1_engine_start.wav", "audio/wav")
print(transcript[:400], "'''")

**단계 6. 대본과 비교해 오차율 재기**

**[3-45] 문자 오차율(CER) 계산**

In [ ]:
import re
def norm(t):
    t = re.sub(r"^(상담원|고객)\s*:", "", t, flags=re.M)     # 화자 표시 제거
    return re.sub(r"[\s.,?!~'\"()\-''']", "", t)              # 공백·문장부호 제거
ref = norm(open("audio/call1_engine_start.txt", encoding="utf-8").read())

### 도면 부품표(BOM) 추출 및 데이터베이스 적재

**단계 7. 도면에서 표제란과 BOM 추출**

**[3-46] 도면 확인**

In [ ]:
display(Image("drawings/drawing_R-DWG-001.png", width=640))

**[3-47] 자연어 지시: 도면 표제란과 BOM 추출**

```
이 도면에서 표제란(도면번호·적용기종·일자)과 부품표(BOM)를 JSON으로 추출하고
SQLite 테이블에 넣어줘.
```

**[3-48] 도면에서 표제란과 BOM 추출**

In [ ]:
dwg = as_json(gemini('''이 도면에서 다음을 JSON으로 추출하세요.
{"도면번호": "", "도면명": "", "적용기종": "", "일자": "",
 "BOM": [{"부품번호": "", "명칭": "", "수량": ""}]}
부품번호가 없는 행은 부품번호를 "(품번 미등록)"으로 적으세요. JSON만 출력하세요.''',
                    "drawings/drawing_R-DWG-001.png", "image/png"))
print(json.dumps(dwg, ensure_ascii=False, indent=1))

**단계 8. SQLite에 적재**

**[3-49] SQLite에 도면 ·BOM 적재**

In [ ]:
import sqlite3

**단계 9. 부품 마스터와 대조**

**[3-50] 부품 마스터와 대조**

In [ ]:
import pandas as pd
master = pd.read_csv("parts/parts_master.csv")
bom = pd.read_sql("SELECT * FROM bom", con)
bom["마스터등록"] = bom["부품번호"].isin(master["부품번호"])
bom[["부품번호", "명칭", "수량", "마스터등록"]]

## 7. 심화 실습: 자연어 질의 기반 데이터베이스 조회

### 실습 안내

**[3-51] 프롬프트: 자연어 질문을 SQL로**

```
당신은 SQLite 데이터베이스를 다루는 분석가입니다.
아래는 이 데이터베이스의 표 구조입니다.
(여기에 표 구조를 붙여 넣습니다)
다음 질문에 답하는 SELECT 문 하나만 출력하세요.
질문: (여기에 현장의 질문을 적습니다)
규칙
- SELECT 문만 출력하고 설명이나 코드 표시는 붙이지 마세요.
- 위 구조에 없는 표나 열은 쓰지 마세요.
- 데이터를 바꾸는 문장(INSERT, UPDATE, DELETE, DROP)은 쓰지 마세요.
- 날짜 비교가 필요하면 저장된 형식에 맞춰 쓰세요.
```

---

실습을 마쳤으면 「파일 → 드라이브에 사본 저장」으로 결과를 남겨 두세요.
막히는 곳은 학습사이트의 같은 절을 함께 보면 설명이 있습니다.